# Phase 1 首个 Mock Turn 验证

这个 notebook 用来验证 Phase 1 当前已经具备的最小假循环：

- 启动本地 Node 服务
- 创建一个 mock session
- 提交一轮玩家输入
- 检查 round、消息流和回放是否正确增加


In [ ]:
# 这个代码块用于准备路径、端口和 HTTP 工具函数，后面的验证都会复用它们。
from pathlib import Path
import json
import os
import subprocess
import time
import urllib.request

repo_root = Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")
server_port = 4318
server_base_url = f"http://127.0.0.1:{server_port}"
server_process = None

def http_get_json(url: str) -> dict:
    # 这个辅助函数用于 GET 一个 JSON 接口。
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read().decode("utf-8"))

def http_post_json(url: str, payload: dict) -> dict:
    # 这个辅助函数用于 POST 一个 JSON 接口。
    request = urllib.request.Request(
        url,
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


In [ ]:
# 这个代码块启动本地服务，并等待健康检查通过。
env = os.environ.copy()
env["PORT"] = str(server_port)

server_process = subprocess.Popen(
    ["node", "--experimental-strip-types", ".\\apps\\server\\src\\server.ts"],
    cwd=repo_root,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
)

health = None
for _ in range(20):
    try:
        health = http_get_json(f"{server_base_url}/api/health")
        break
    except Exception:
        time.sleep(0.5)

assert health is not None, "服务启动失败，健康检查没有通过"
print("健康检查结果:", health)


In [ ]:
# 这个代码块创建一个 mock session，作为后续提交 turn 的起点。
create_payload = {
    "ruleDirectoryName": "VHS",
    "storyDirectoryName": "The_Silence",
    "locale": "zh-CN",
    "playMode": "single_player",
    "gmArchitecture": "single_agent",
    "modelAccessMode": "mock",
    "debugEnabled": True,
    "promptDebugEnabled": False,
    "logViewMode": "compact",
}

created = http_post_json(f"{server_base_url}/api/sessions", create_payload)
session_id = created["session"]["id"]

print("Session ID:", session_id)
print("初始 round:", created["session"]["currentRound"])
print("初始消息数:", len(created["messages"]))

assert created["session"]["currentRound"] == 0
assert len(created["messages"]) >= 2


In [ ]:
# 这个代码块提交第一轮玩家输入，并检查系统是否正确追加 mock 主持回应。
turn_payload = {
    "playerInput": "我先检查录像厅入口，再轻声询问莉莉她姐姐最后一次出现在哪里。"
}

turned = http_post_json(f"{server_base_url}/api/sessions/{session_id}/turns", turn_payload)
last_message = turned["messages"][-1]

print("当前 round:", turned["session"]["currentRound"])
print("当前消息数:", len(turned["messages"]))
print("当前回放数:", len(turned["replay"]))
print("最后一条消息类型:", last_message["kind"])
print("\n最后一条消息内容:\n")
print(last_message["content"])

assert turned["session"]["currentRound"] == 1
assert len(turned["messages"]) >= 4
assert len(turned["replay"]) >= 6
assert last_message["kind"] == "gm_narration"


In [ ]:
# 这个代码块读取更新后的 session，再次确认 storyFlags 和回放记录已经写入。
fetched = http_get_json(f"{server_base_url}/api/sessions/{session_id}")
story_flags = fetched["session"]["gameState"]["storyFlags"]
replay_types = [item["type"] for item in fetched["replay"][-3:]]

print("最近记录的玩家输入:", story_flags.get("last_player_input"))
print("最近三个回放类型:", replay_types)

assert story_flags.get("last_mock_turn_round") == 1
assert "turn_submitted" in replay_types


In [ ]:
# 这个代码块用于清理后台服务进程，建议在验证完成后执行一次。
if server_process is not None and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_process.kill()

print("服务已停止")
